# Introduction to pandas

In [ ]:
import pandas as pd

trade = pd.read_csv(
    "../../data/trade_summary.csv")
trade.info()

## DataFrames and Series

In [ ]:
exports = trade["exports_usd_m"]
print(type(trade))
print(type(exports))
exports.describe().round(1)

## Selecting data

In [ ]:
cols = ["reporter", "year", "exports_usd_m"]
subset = trade[cols]
print(subset.head(3))

row = trade.loc[0]
print(row["reporter"], row["year"])
print(trade.iloc[0:2, 0:3])

## Filtering records

In [ ]:
africa = trade[trade["region"] == "Africa"]
recent_agr = trade[
    (trade["year"] >= 2022)
    & (trade["product_code"] == "AGR")
]
print(len(africa), len(recent_agr))

big = trade.query("exports_usd_m > 500000")
big[["reporter", "year", "product_code"]]

## Sorting

In [ ]:
top = (
    trade[trade["year"] == 2023]
    .sort_values("exports_usd_m",
                 ascending=False)
    .head(5)
)
top[["reporter", "exports_usd_m"]]

## Calculated fields

In [ ]:
trade["balance_usd_m"] = (
    trade["exports_usd_m"]
    - trade["imports_usd_m"]
)
trade["exports_usd_bn"] = (
    trade["exports_usd_m"] / 1000
).round(1)
trade["surplus"] = trade["balance_usd_m"] > 0
trade[["reporter", "balance_usd_m",
       "surplus"]].head(3)

## Grouping

In [ ]:
by_region = (
    trade.groupby(["region", "year"])
    ["exports_usd_m"]
    .sum()
    .reset_index()
)
by_region.head()

In [ ]:
summary = trade.groupby("reporter").agg(
    total=("exports_usd_m", "sum"),
    average=("exports_usd_m", "mean"),
    years=("year", "nunique"),
)
summary.sort_values("total",
                    ascending=False).head()

## Pivot tables and growth

In [ ]:
pivot = trade.pivot_table(
    index="reporter", columns="year",
    values="exports_usd_m", aggfunc="sum")
growth = pivot.pct_change(axis=1) * 100
growth.round(1).head(6)

## Combining with reference data (preview of Module 07)

In [ ]:
countries = pd.read_excel(
    "../../data/countries.xlsx")
merged = trade.merge(
    countries[["iso3", "income_group"]],
    left_on="reporter_iso3",
    right_on="iso3", how="left")
merged.groupby("income_group")[
    "exports_usd_m"].sum()